# From raw reviews to governed insights — a medallion pipeline in SQL

**The story:** you run a food-delivery app in 6 countries. Customers leave reviews in 6 languages — English, Hindi, Spanish, German, Japanese, Portuguese — sometimes with names and phone numbers inside the text. Operations wants **English text, sentiment, issue category and food item** in clean, governed tables — with quality checks and cost control.

**What this notebook builds** (25 raw reviews):

| Layer | Object | What it holds |
| --- | --- | --- |
| BRONZE | `bronze_delivery_reviews` | raw reviews as-is: multilingual text, PII, junk, duplicates |
| SILVER | `dq_rejected_reviews` (view) | reviews rejected by quality rules, with the reason |
| SILVER | `silver_delivery_reviews_masked` | PII-masked text, fixed types, email column dropped |
| SILVER | `silver_delivery_reviews_enriched` | the AI output: English text, sentiment, issue type, food item |
| GOLD | `gold_delivery_issue_trend` (view) | issue counts and ratings by country |
| GOLD | `gold_support_escalation_queue` | 5 worst reviews + AI-drafted apology replies |

**AI functions used in this pipeline, and where:**

| Function | Step | Usage |
| --- | --- | --- |
| `ai_similarity` | Silver — fuzzy DQ | near-duplicate detection on candidate pairs |
| `ai_mask` | Silver — PII masking | hide names, phones, emails before any model call |
| `ai_translate` | Silver — enrichment | `review_en`: every review in English |
| `ai_analyze_sentiment` | Silver — enrichment | `sentiment` signal |
| `ai_classify` | Silver — enrichment | `issue_type`, routed to ops teams |
| `ai_extract` | Silver — enrichment | `food_item` |
| `ai_summarize` | Silver — enrichment | `review_summary`: 15-word digest for dashboards |
| `ai_fix_grammar` | Silver — enrichment | `review_en_polished`: clean text for customer-facing use |
| `ai_gen` | Gold — escalation queue | drafted apology replies (5 rows, cost-capped) |

**What each bronze column means:**

| Column | Description |
| --- | --- |
| `review_id`, `order_id` | identifiers |
| `review_ts` | when the review was submitted (raw STRING in bronze — silver casts to TIMESTAMP) |
| `city`, `country`, `language`, `rating` | review metadata |
| `review_text` | the free text we process — the whole point of this demo |
| `customer_email` | PII column — stays in bronze, never flows downstream |
| `expected_sentiment`, `expected_issue` | hand-labeled ground truth for the quality check at the end |

**Cost control — the three golden rules used in this notebook:**
1. Call AI functions **once**, writing results into a table — never inside dashboards or repeated queries
2. **Mask PII before** sending text to any model
3. **Filter and limit** rows before calling AI (we draft apology replies for only the 5 worst reviews)

Run top to bottom. Every step is idempotent (`CREATE OR REPLACE`) — safe to re-run — and every step ends with its own **verification cell**, so you can confirm each result before moving on.
Runtime: ~10–15 minutes. Works on a serverless SQL warehouse or serverless compute.

In [0]:
%sql
-- One schema for the whole demo: workspace.food_delivery_demo
-- Everything we create lives here, and the cleanup cell at the end drops it in one go.
CREATE SCHEMA IF NOT EXISTS workspace.food_delivery_demo;
USE workspace.food_delivery_demo;

In [0]:
%sql
-- BRONZE: raw reviews exactly as they "landed" — no cleaning, no masking, raw STRING timestamps.
-- This mimics data arriving from an app database, a CSV drop, or a CDC feed.
-- expected_* columns are hand-labeled ground truth, used for the quality check at the end
-- (in real life you would label a small sample of rows).
CREATE OR REPLACE TABLE bronze_delivery_reviews AS
SELECT * FROM VALUES
  ( 1,'O1001','2026-11-01 12:10','London','UK','en',1,'Food arrived 70 minutes late and completely cold. Very disappointed.','c01@ex.com','negative','delivery_delay'),
  ( 2,'O1002','2026-11-01 13:05','Austin','US','en',5,'Amazing biryani, still hot and the rider was super friendly!','c02@ex.com','positive','none'),
  ( 3,'O1003','2026-11-01 19:40','Toronto','CA','en',2,'Ordered two burgers but only got one. Want a refund please.','c03@ex.com','negative','wrong_or_missing_item'),
  ( 4,'O1004','2026-11-02 20:15','Chicago','US','en',3,'Tastes great but the delivery fee is way too high now.','c04@ex.com','negative','price'),
  ( 5,'O1005','2026-11-02 21:00','Miami','US','en',3,'ok','c05@ex.com','neutral','none'),
  ( 6,'O1006','2026-11-03 12:30','Austin','US','en',2,'The app kept crashing every time I tried to pay.','c06@ex.com','negative','app_issue'),
  ( 7,'O1007','2026-11-03 18:45','Austin','US','en',5,'Amazing biryani, still hot and the rider was super friendly!','c07@ex.com','positive','none'),
  ( 8,'O1008','2026-11-03 19:10','Mumbai','IN','hi',1,'Khana ek ghante late aaya aur bilkul thanda tha.','c08@ex.com','negative','delivery_delay'),
  ( 9,'O1009','2026-11-04 13:15','Delhi','IN','hi',5,'Biryani bahut swadisht thi, rider bhi bahut accha tha!','c09@ex.com','positive','none'),
  (10,'O1010','2026-11-04 14:00','Pune','IN','hi',2,'Maine do pizza order kiye the, sirf ek mila.','c10@ex.com','negative','wrong_or_missing_item'),
  (11,'O1011','2026-11-04 20:30','Delhi','IN','hi',2,'Khana bilkul kharab tha, taste achha nahi laga.','c11@ex.com','negative','food_quality'),
  (12,'O1012','2026-11-05 12:20','Mexico City','MX','es',1,'La comida llegó una hora tarde y fría.','c12@ex.com','negative','delivery_delay'),
  (13,'O1013','2026-11-05 13:10','Madrid','ES','es',5,'¡La pizza estaba increíble y llegó caliente!','c13@ex.com','positive','none'),
  (14,'O1014','2026-11-05 19:55','Guadalajara','MX','es',2,'Pedí dos tacos y solo llegó uno. Quiero mi dinero.','c14@ex.com','negative','wrong_or_missing_item'),
  (15,'O1015','2026-11-06 12:05','Bogota','CO','es',3,'El costo de envío es demasiado alto.','c15@ex.com','negative','price'),
  (16,'O1016','2026-11-06 13:00','Berlin','DE','de',1,'Das Essen kam eine Stunde zu spät und kalt an.','c16@ex.com','negative','delivery_delay'),
  (17,'O1017','2026-11-06 19:30','Munich','DE','de',5,'Sehr leckeres Essen und schnelle Lieferung!','c17@ex.com','positive','none'),
  (18,'O1018','2026-11-06 20:10','Hamburg','DE','de',2,'Ich habe zwei Burger bestellt, aber nur einen bekommen.','c18@ex.com','negative','wrong_or_missing_item'),
  (19,'O1019','2026-11-07 12:15','Berlin','DE','de',1,'Rufen Sie mich unter 0176 1234567 an. Das Essen war kalt und hat furchtbar geschmeckt.','c19@ex.com','negative','food_quality'),
  (20,'O1020','2026-11-07 12:40','Tokyo','JP','ja',1,'料理が1時間遅れて届き、冷めていました。','c20@ex.com','negative','delivery_delay'),
  (21,'O1021','2026-11-07 13:25','Osaka','JP','ja',5,'とても美味しかったです!スムーズな配達でした。','c21@ex.com','positive','none'),
  (22,'O1022','2026-11-07 18:50','Tokyo','JP','ja',2,'注文した2つのバーガーのうち1つしか届きませんでした。','c22@ex.com','negative','wrong_or_missing_item'),
  (23,'O1023','2026-11-07 19:05','Sao Paulo','BR','pt',1,'A comida chegou uma hora atrasada e fria.','c23@ex.com','negative','delivery_delay'),
  (24,'O1024','2026-11-07 19:40','Sao Paulo','BR','pt',5,'Muito saboroso, entrega rápida e pontual!','c24@ex.com','positive','none'),
  (25,'O1025','2026-11-07 20:00','Lisbon','PT','pt',1,'Meu nome é Ana Silva, ligue para 21 98765-4321. O pedido nunca chegou.','c25@ex.com','negative','delivery_delay')
AS t(review_id, order_id, review_ts, city, country, language, rating, review_text, customer_email, expected_sentiment, expected_issue);

In [0]:
%sql
-- Sanity check: all reviews as loaded — raw multilingual text, PII, junk and duplicates included.
-- Expect 25 rows: the junk row is id 5 ('ok'), the duplicate is id 7.
SELECT *
FROM   bronze_delivery_reviews
ORDER  BY review_id;

In [0]:
%sql
-- Eyeball the raw data: different languages, the junk row (id 5: 'ok'),
-- the duplicate rows (ids 2 and 7), and the PII rows (ids 19 and 25).
SELECT review_id, language, rating, review_text
FROM   bronze_delivery_reviews
ORDER  BY review_id
LIMIT 8;

## SILVER — step 1: data-quality rules

Junk rows burn AI money: a 2-character "ok" and a copy-pasted duplicate are not worth a model call. We define rejections as a **view** — visible, auditable, re-runnable, stores nothing — and exclude them from silver.

**Our two rules:**
1. `too_short` — text shorter than 5 characters carries no signal
2. `duplicate_text` — the same text posted twice; keep the *first* review only

The view records which review it kept (`keep_id` = first occurrence), so re-runs behave deterministically.

In [0]:
%sql
-- Rejected reviews are defined as a VIEW: auditable, re-runnable, stores nothing.
-- Rule 1 (too_short):      text shorter than 5 characters carries no signal.
-- Rule 2 (duplicate_text):  same text posted twice — keep the FIRST review only.
CREATE OR REPLACE VIEW dq_rejected_reviews AS
WITH duplicates AS (
  SELECT min(review_id) AS keep_id, review_text
  FROM   bronze_delivery_reviews
  GROUP  BY review_text
  HAVING count(*) > 1
)
SELECT b.review_id,
       CASE WHEN length(trim(b.review_text)) < 5 THEN 'too_short'
            ELSE 'duplicate_text' END AS reject_reason
FROM   bronze_delivery_reviews b
LEFT   JOIN duplicates d ON b.review_text = d.review_text
WHERE  length(trim(b.review_text)) < 5                        -- rule 1: junk
   OR  (d.keep_id IS NOT NULL AND b.review_id <> d.keep_id);  -- rule 2: duplicates

In [0]:
%sql
-- Expect exactly 2 rows: review 5 ('ok' -> too_short) and review 7 (copy of review 2 -> duplicate_text).
SELECT * FROM dq_rejected_reviews
ORDER  BY review_id;

## SILVER — step 1b: near-duplicates — `ai_similarity`

The DQ view catches **exact** duplicates (same text). Real data also has **near**-duplicates: "Food arrived cold and late" vs "My meal came cold and very late" — different strings, same complaint. `ai_similarity` scores any two texts from 0.0 to 1.0.

**Cost control:** every pair is a model call — never cross-join a whole table (25 reviews = 300 calls). Run it on candidate pairs only: same order, same city+day, or flagged by cheap signals like the `text_hash` fingerprint.

In [0]:
%sql
-- 2 AI calls total: the exact-duplicate pair (reviews 2 vs 7) should score high,
-- the different pair (1 vs 2) low. This is the candidate-pairs pattern, not a cross join.
SELECT 'duplicates: 2 vs 7' AS pair,
       round(ai_similarity(max(CASE WHEN review_id = 2 THEN review_text END),
                           max(CASE WHEN review_id = 7 THEN review_text END)), 2) AS similarity
FROM   bronze_delivery_reviews
UNION ALL
SELECT 'different: 1 vs 2',
       round(ai_similarity(max(CASE WHEN review_id = 1 THEN review_text END),
                           max(CASE WHEN review_id = 2 THEN review_text END)), 2)
FROM   bronze_delivery_reviews;

## SILVER — step 2: mask PII **before** any AI call

`ai_mask` replaces entities (names, phone numbers, email addresses) with `[MASKED]`, so customer identities never reach a model. In the same step we:

* **cast** `review_ts` from raw STRING to a real TIMESTAMP (bronze stays raw; silver fixes types)
* **fingerprint** the raw text with `sha2` — a cheap hash for future deduplication
* **drop** the `customer_email` column — raw identifiers never leave bronze

Governance rule: only masked text flows downstream.

In [0]:
%sql
-- Mask PII, fix types, drop the email column.
-- Only 23 rows survive: 25 raw minus 2 rejected by the DQ view.
CREATE OR REPLACE TABLE silver_delivery_reviews_masked AS
SELECT
  review_id, order_id,
  CAST(review_ts AS TIMESTAMP) AS review_ts,   -- bronze stored raw strings; silver gets real types
  city, country, language, rating,
  sha2(review_text, 256) AS text_hash,          -- fingerprint for future dedup
  ai_mask(review_text, ARRAY('person name', 'phone number', 'email address')) AS review_text_masked
FROM   bronze_delivery_reviews
WHERE  review_id NOT IN (SELECT review_id FROM dq_rejected_reviews);

In [0]:
%sql
-- 25 raw rows - 2 rejected = 23 clean rows expected.
-- PII is now [MASKED], the email column is gone, review_ts is a real TIMESTAMP.
SELECT *
FROM   silver_delivery_reviews_masked
ORDER  BY review_id;

In [0]:
%sql
-- Rows 19 (German) and 25 (Portuguese) contained phone numbers and a name — confirm they are now [MASKED].
SELECT review_id, review_text_masked
FROM   silver_delivery_reviews_masked
WHERE  review_id IN (19, 25);

## SILVER — step 3: the magic — 6 AI functions, **one query**

Every clean review gets, in a single SELECT:

| AI function | Output column | What it gives ops |
| --- | --- | --- |
| `ai_translate(text, 'en')` | `review_en` | every analyst can read every review |
| `ai_analyze_sentiment(text)` | `sentiment` | positive / negative / neutral signal |
| `ai_classify(text, ARRAY(...))` | `issue_type` | routes the review to the right team |
| `ai_extract(text, '["food_item"]')` | `food_item` | which dish the review is about |
| `ai_summarize(text, 15)` | `review_summary` | 15-word digest for dashboards |
| `ai_fix_grammar(text)` | `review_en_polished` | clean text, safe for customer-facing use |

**Prompt design in action:** the `ai_classify` labels are short, non-overlapping, and map 1:1 to ops teams (`delivery_delay`, `food_quality`, `wrong_or_missing_item`, `price`, `app_issue`, `none`). Vague or overlapping labels make the model guess.

**One pass, one payment:** a CTE translates + scores every review first; the outer SELECT then summarizes and polishes the *already-translated* `review_en` — so nothing is translated twice.

Results are **stored in the table** — this is the only time we pay for these AI calls. Never re-run AI in dashboards.

In [0]:
%sql
-- THE demo moment: 6 AI functions, one query, results stored forever.
-- Inner pass (CTE):  translate, sentiment, classify, extract.
-- Outer pass:        ai_summarize + ai_fix_grammar reuse the CTE's review_en — nothing is translated twice.
-- ai_extract returns a VARIANT — ':response.food_item.value::STRING' pulls the field out.
CREATE OR REPLACE TABLE silver_delivery_reviews_enriched AS
WITH base AS (
  SELECT
    m.review_id, m.order_id, m.review_ts, m.city, m.country, m.language, m.rating,
    m.review_text_masked,
    ai_translate(m.review_text_masked, 'en')   AS review_en,
    ai_analyze_sentiment(m.review_text_masked) AS sentiment,
    ai_classify(m.review_text_masked,
      ARRAY('delivery_delay','food_quality','wrong_or_missing_item','price','app_issue','none')) AS issue_type,
    ai_extract(m.review_text_masked, '["food_item"]', MAP('version','2.1'))
      :response.food_item.value::STRING AS food_item
  FROM silver_delivery_reviews_masked m
)
SELECT b.*,
  ai_summarize(b.review_en, 15) AS review_summary,    -- short digest for dashboards
  ai_fix_grammar(b.review_en)   AS review_en_polished,  -- clean text for customer-facing use
  current_timestamp()           AS processed_at        -- when the AI ran (audit trail)
FROM base b;

In [0]:
%sql
-- One row per language family: English original (1), Hindi (8), German (16), Japanese (20).
-- Everything is now English + structured: sentiment, issue_type, food_item,
-- plus the two new columns: a 15-word summary and the polished English text.
SELECT review_id, language, sentiment, issue_type, food_item, review_en,
       review_summary, review_en_polished
FROM   silver_delivery_reviews_enriched
WHERE  review_id IN (1, 8, 16, 20);

## GOLD — business-ready insights

* `gold_delivery_issue_trend` (**VIEW**): where do which problems happen? Reads silver on every query — always fresh, **zero AI calls**.
* `gold_support_escalation_queue` (**TABLE**): the 5 worst reviews with an **AI-drafted apology reply** — `ai_gen` runs on 5 rows only, not all 25. Cost control in action. It is a TABLE on purpose: a VIEW would re-draft the replies (and re-pay) on every read.

Both are documented, English, PII-free — ready for BI and AI consumption.

In [0]:
%sql
-- Business insight #1: issue volume and severity by country.
-- A VIEW over stored silver data — no AI calls, always fresh.
CREATE OR REPLACE VIEW gold_delivery_issue_trend AS
SELECT country, issue_type,
       count(*)              AS reviews,
       round(avg(rating), 1) AS avg_rating
FROM   silver_delivery_reviews_enriched
WHERE  issue_type <> 'none'      -- reviews with no issue are not actionable
GROUP  BY country, issue_type;

In [0]:
%sql
-- Expect issues concentrated in delivery_delay and wrong_or_missing_item.
SELECT * FROM gold_delivery_issue_trend
ORDER  BY reviews DESC, country
LIMIT 10;

In [0]:
%sql
-- Business insight #2: draft apology replies for the worst reviews.
-- WHERE + ORDER BY + LIMIT = cost control: 5 ai_gen calls instead of 25.
-- A TABLE (not VIEW) so drafts are frozen — regenerating on every read would re-pay.
CREATE OR REPLACE TABLE gold_support_escalation_queue AS
SELECT review_id, country, issue_type, rating, review_en,
       ai_gen(
         'Write a polite 2-sentence apology to a food-delivery customer. '
         || 'Their problem: ' || issue_type || '. '
         || 'Their feedback: ' || review_en || ' '
         || 'Do not promise a refund or compensation.'     -- guardrail in the prompt
       ) AS draft_reply
FROM   silver_delivery_reviews_enriched
WHERE  sentiment = 'negative' AND rating <= 2
ORDER  BY rating, review_id
LIMIT 5;

In [0]:
%sql
-- 5 rows: the review, the problem, the customer feedback in English, and the AI-drafted reply.
SELECT * FROM gold_support_escalation_queue;

## Quality checks — is the AI any good?

Bronze secretly carried `expected_sentiment` / `expected_issue` labels. A tiny labeled sample is the cheapest quality check there is — we compare the AI output against it in two steps:

1. **Overall accuracy** — how often the AI matched the labels
2. **Accuracy by language** — catches a weak language that the overall number hides

If a language scores badly, you know exactly what to fix: better labels, more examples, or a stronger model via `ai_query`.

In [0]:
%sql
-- How often did the AI output match the hand-labeled ground truth?
SELECT count(*) AS reviews_checked,
       round(avg(CASE WHEN e.sentiment = b.expected_sentiment THEN 1.0 ELSE 0 END) * 100, 1) AS sentiment_accuracy_pct,
       round(avg(CASE WHEN e.issue_type  = b.expected_issue    THEN 1.0 ELSE 0 END) * 100, 1) AS issue_accuracy_pct
FROM   silver_delivery_reviews_enriched e
JOIN   bronze_delivery_reviews b USING (review_id);

In [0]:
%sql
-- Drill-down: the overall number can hide a weak language.
SELECT e.language, count(*) AS n,
       round(avg(CASE WHEN e.sentiment = b.expected_sentiment THEN 1.0 ELSE 0 END) * 100, 1) AS sentiment_accuracy_pct
FROM   silver_delivery_reviews_enriched e
JOIN   bronze_delivery_reviews b USING (review_id)
GROUP  BY e.language
ORDER  BY n DESC;

## Cost control recap

1. **Enrich once, reuse forever** — the cell below reads *stored* columns; zero model calls. Never put AI functions in BI queries, `WHERE` clauses, or views that re-execute on every read.
2. **Filter before you call** — the escalation queue drafted replies for 5 rows, not 25.
3. **Mask first** — cheaper *and* safer: less sensitive data goes to the models.
4. **Re-run the pipeline only when inputs change** — `CREATE OR REPLACE` makes re-runs safe, but every re-run re-pays the AI calls.
5. Monitor spend in the workspace **usage / billing** dashboards.

In [0]:
%sql
-- pure SQL on stored results: instant, no model calls
SELECT issue_type, count(*) AS reviews
FROM   silver_delivery_reviews_enriched
GROUP  BY issue_type
ORDER  BY reviews DESC;

## Optional: going further with `ai_query` (not run in this demo)

Full control — custom prompt, chosen model, **structured output** via `responseFormat`. Costs more and needs Model Serving access. The pattern:

```sql
CREATE OR REPLACE TABLE silver_delivery_reviews_advanced AS
SELECT review_id,
       ai_query(
         'system.ai.meta-llama-3-3-70b-instruct',
         'You are a review analyst. Review: ' || review_text_masked,
         responseFormat => 'STRUCT<sentiment:STRING, severity:INT, needs_refund:BOOLEAN>'
       ) AS insight
FROM   silver_delivery_reviews_masked;

SELECT review_id, insight.sentiment, insight.severity FROM silver_reviews_advanced;
```

In production you would add enum guards on the output, a prompt-version column, `failOnError => false` for retries, and an audit table — see the session deck for the checklist.

## Governance — per layer, all the way down

Governance is not a final step — each layer has its own contract:

| Layer | Governance concern | What we do here |
| --- | --- | --- |
| BRONZE | raw PII (emails, names, phones) | keep raw truth, but tag `contains_pii=true` and restrict access — nothing raw flows downstream |
| SILVER | safe to share | `ai_mask` before any model call, drop `customer_email`, fix types, document every column's meaning |
| GOLD | business-ready | commented, tagged, English-only, PII-free — ready for BI and AI consumption |

Below we apply **comments** (Catalog Explorer, autocomplete tooltips, Genie answers) and **tags** (layer, domain, PII flag — for search, audits, access policies) to every table, layer by layer.

The same Unity Catalog toolset also offers **row filters**, **column masks** and **lineage** — worth exploring after the session.

In [0]:
%sql
-- Table comments, layer by layer — power Catalog Explorer, tooltips, and Genie answers.
COMMENT ON TABLE bronze_delivery_reviews IS
  'Raw multilingual food-delivery reviews, as landed: PII, junk and duplicates included. expected_sentiment/expected_issue are hand-labeled ground truth for the quality check.';
COMMENT ON TABLE silver_delivery_reviews_masked IS
  'Clean reviews: PII masked with ai_mask, email dropped, types fixed. 23 of 25 rows survive the DQ rules.';
COMMENT ON TABLE silver_delivery_reviews_enriched IS
  'AI-enriched reviews: English translation, polished grammar, 15-word summary, sentiment, issue type, food item. Built with Databricks SQL AI Functions.';
COMMENT ON TABLE gold_support_escalation_queue IS
  'Five worst reviews with AI-drafted apology replies for the support team.';

In [0]:
%sql
-- Tags, layer by layer: a layer tag on each table, domain on all, PII flag only on bronze.
ALTER TABLE bronze_delivery_reviews          SET TAGS ('layer' = 'bronze', 'domain' = 'voc', 'contains_pii' = 'true');
ALTER TABLE silver_delivery_reviews_masked   SET TAGS ('layer' = 'silver', 'domain' = 'voc', 'contains_pii' = 'false');
ALTER TABLE silver_delivery_reviews_enriched SET TAGS ('layer' = 'silver', 'domain' = 'voc', 'contains_pii' = 'false');
ALTER TABLE gold_support_escalation_queue     SET TAGS ('layer' = 'gold',   'domain' = 'voc', 'contains_pii' = 'false');

In [0]:
%sql
-- Comments are visible in Catalog Explorer and query autocomplete tooltips.
-- Expect one row per table — every layer documented.
SELECT table_name, comment
FROM   workspace.information_schema.tables
WHERE  table_schema = 'food_delivery_demo'
  AND  table_name IN ('bronze_delivery_reviews', 'silver_delivery_reviews_masked',
                      'silver_delivery_reviews_enriched', 'gold_support_escalation_queue');

In [0]:
%sql
-- Tags are queryable through the catalog information schema.
-- Expect 12 rows: 4 tables x 3 tags each (layer, domain, contains_pii — bronze = true).
SELECT table_name, tag_name, tag_value
FROM   workspace.information_schema.table_tags
WHERE  schema_name = 'food_delivery_demo'
ORDER  BY table_name, tag_name;

## Wrap-up

* **Bronze** — 25 raw multilingual reviews landed as-is (PII, junk, duplicates included)
* **Silver** — quality rules (exact + fuzzy duplicates via `ai_similarity`), PII masking, then **one query, 6 functions**: translate, sentiment, classify, extract, summarize, fix-grammar
* **Gold** — `gold_delivery_issue_trend` view + cost-capped `gold_support_escalation_queue` with AI-drafted replies
* **Quality** — measured against labels · **Cost** — capped by design · **Governance** — UC comments & tags, applied per layer

**Production next steps** (beyond this demo): schedule the notebook as a Lakeflow Job, add incremental loading, alerting on quality scores, and prompt-version columns.

**Cleanup:** uncomment and run the cell below to remove everything. It is commented out so "Run All" does not wipe your demo mid-presentation.

In [0]:
%sql
-- Removes every table and view created by this demo.
-- DROP SCHEMA IF EXISTS workspace.food_delivery_demo CASCADE;